Author: Hui Fang\
Purpose: ST 554 Final Project\
Date: 4/20/2026

# Introction

This project demonstrates the end-to-end use of `PySpark` for both machine learning and real-time data processing. The data is adapted from the [UCI machine learning repository](https://archive.ics.uci.edu/dataset/849/power+consumption+of+tetouan+city), which examines how power consumption across different zones of Tetouan city relates to various factors such as time of day, temperature, and humidity.\
The project consistsof two components. The first component focuses on building an **elastic net regression model** using `Spark MLlib`, incorporating SQL-based feature engineering, one-hot encoding, PCA, and cross-validated hyperparameter tuning. The second component extends this work into a streaming context: a `Python` script generates incoming CSV files, `Spark` monitors the folder as a stream, applies the fitted model to produce predictions and residuals in real time, and outputs the results to the console.\
 Together, these components highlight Spark's ability to unify batch modeling and streaming inference within a single, conherent workflow.

# Part I: Fitting Model

First let's import some of the modules needed and create a Spark session.

In [2]:
# import modules needed
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, VectorAssembler, PCA
from pyspark.ml import Pipeline
# Create a Spark session
spark = SparkSession.builder.getOrCreate()

## 1. Read in data and convert to Spark DataFrame

In [7]:
# read in data
power_pd = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")
power_pd.head()
# convert to Spark Dataframe
power_sdf = spark.createDataFrame(power_pd)
# print out the data frame schema
power_sdf.printSchema()
# check the spark DataFrame
power_sdf.show(10)

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.0

## 2. Cast `Hour` to Double using SQLTransformer

The printed schema indicates that `Hour` is stored as a long. To ensure compatibility with downstream `MLlib` transformations, we cast `Hour` to `DoubleType` using an `SQLTransformer` within the pipeline.

In [10]:
# cast Hour vaiable to double
hour_cast_sql = SQLTransformer(statement = "SELECT *, CAST(Hour AS DOUBLE) AS Hour_double FROM __THIS__")

## 3. Binarize the `Hour` column

Next, we create a binary indicator for the `Hour` variable, where values less than 6.5 represent nighttime and values greater than or equal to 6.5 represent daytime.

In [ ]:
# use Binarizer on Hour_double
hour_bin = Binarizer(
    inputCol = "Hour_double",
    outputCol = "Daytime",
    threshold = 6.5)

